In [0]:
import pyspark.pipelines as dp
from pyspark.sql.functions import window, col, sum, max, count, approx_count_distinct

In [0]:
# -------------------------------------------------------------------------
# 1. GOLD STREAMING TABLE (Fine-Grained 1-Minute Aggregations)
# Continuously ingests from Silver and outputs streaming window updates
# -------------------------------------------------------------------------
@dp.table(
    name="streaming_demo.gold.wiki_edits_per_minute",
    comment="Near-real-time 1-minute windowed edit counts per wiki"
)
def gold_wiki_edits_1m():
    return (
        dp.read_stream("streaming_demo.silver.wiki_events_slv")
        .withWatermark("event_ts", "2 minutes")
        .groupBy(window(col("event_ts"), "1 minute"), col("wiki"))
        .agg(
            count("*").alias("edit_count"),
            approx_count_distinct("user").alias("unique_editors")
        )
        .selectExpr(
            "window.start as window_start",
            "window.end as window_end",
            "wiki",
            "edit_count",
            "unique_editors"
        )
    )

In [0]:
# -------------------------------------------------------------------------
# 2. GOLD MATERIALIZED VIEW (Built on top of Gold Streaming Table)
# Full recompute / incremental refresh for executive daily reporting
# -------------------------------------------------------------------------
@dp.materialized_view(
    name="streaming_demo.gold.gold_wiki_edits_mv",
    comment="Daily aggregated rollup built on top of gold_wiki_edits_1m for Power BI executive views"
)
def gold_wiki_edits_daily_summary():
    return (
        dp.read("streaming_demo.gold.wiki_edits_per_minute")  # dp.read performs batch read over the Gold streaming table
        .withColumn("edit_date", col("window_start").cast("date"))
        .groupBy("edit_date", "wiki")
        .agg(
            sum("edit_count").alias("total_daily_edits"),
            max("unique_editors").alias("peak_unique_editors_per_min")
        )
    )